In [1]:
import polars as pl

In [2]:
df = pl.read_parquet('data/NF-CSE-CIC-IDS2018-V2.parquet',)
df_shape = df.shape
print('Форма датасета:', df_shape)
print('Типы колонок (последние пять:)', df.dtypes[-5:])
print(df.select(pl.col('Label').n_unique().alias('Уникальные значения в колонке Label')))
print(df.select(pl.col('Attack').n_unique().alias('Уникальные значения в колонке Attack')))

Форма датасета: (17129715, 43)
Типы колонок (последние пять:) [Int16, Int32, Int8, Int8, String]
shape: (1, 1)
┌─────────────────────────────────┐
│ Уникальные значения в колонке … │
│ ---                             │
│ u32                             │
╞═════════════════════════════════╡
│ 2                               │
└─────────────────────────────────┘
shape: (1, 1)
┌─────────────────────────────────┐
│ Уникальные значения в колонке … │
│ ---                             │
│ u32                             │
╞═════════════════════════════════╡
│ 15                              │
└─────────────────────────────────┘


In [3]:
print('Записей помечено как Label == 0 ("Benign") %:', df.filter(pl.col('Label') == 0).shape[0] / df_shape[0] * 100)
print('Записей помечено как Label == 1 ("Attack") %:', df.filter(pl.col('Label') == 1).shape[0] / df_shape[0] * 100)

Записей помечено как Label == 0 ("Benign") %: 88.1607487339982
Записей помечено как Label == 1 ("Attack") %: 11.8392512660018


In [4]:
df = df.with_columns(pl.when(pl.col('Label') != 0)
                     .then(1)
                     .otherwise(0)
                     .alias('is_attack')
)

In [5]:
df_aggr = df.filter(pl.col('Label') != 0).group_by(pl.col('Attack')).agg(
    pl.col('FLOW_DURATION_MILLISECONDS').mean(), 
    pl.col('IN_BYTES').mean().alias('avg_in_bytes'), 
    pl.col('Label').count().alias('Count')
).sort(pl.col('avg_in_bytes'), descending=True)

display(df_aggr.head(3))

df_aggr.write_parquet('data/attack_summary_by_type.parquet', compression='zstd')

Attack,FLOW_DURATION_MILLISECONDS,avg_in_bytes,Count
str,f64,f64,u32
"""DDOS attack-LOIC-UDP""",4.1959e6,5.8540e6,2112
"""DDoS attacks-LOIC-HTTP""",3.7241e6,25991.904925,207078
"""Brute Force -XSS""",3.7568e6,16871.281553,927


In [18]:
print('Общее распределение по PROTOCOL')
total_proto = df['PROTOCOL'].count()
display(df.group_by(pl.col('PROTOCOL')).agg((pl.col('Label').count() / total_proto * 100).alias('Total protocol percentage')))

print('Распределение по PROTOCOL только для Benign')
df_filtered = df.filter(pl.col('Label') == 0)
total_proto = df_filtered['PROTOCOL'].count()
df_benign = df_filtered.group_by(pl.col('PROTOCOL')).agg((pl.col('Label').count() / total_proto * 100).alias('Benign protocol percentage'))
display(df_benign)

print('Распределение по PROTOCOL только для Attack')
df_filtered = df.filter(pl.col('Label') != 0)
total_proto = df_filtered['PROTOCOL'].count()
df_atack = df_filtered.group_by(pl.col('PROTOCOL')).agg((pl.col('Label').count() / total_proto * 100).alias('Attack protocol percentage'))
display(df_atack)

print('Сравнение PROTOCOL между Benign и Attack')
df_benign.join(df_atack, on='PROTOCOL', how='left')

Общее распределение по PROTOCOL


PROTOCOL,Total protocol percentage
i8,f64
6,54.561836
58,0.00488
1,0.028354
17,45.399214
2,0.005698
47,0.000018


Распределение по PROTOCOL только для Benign


PROTOCOL,Benign protocol percentage
i8,f64
6,49.046825
58,0.005536
1,0.030043
17,50.911729
2,0.005847
47,0.00002


Распределение по PROTOCOL только для Attack


PROTOCOL,Attack protocol percentage
i8,f64
6,95.629256
1,0.015779
2,0.004586
17,4.350379


Сравнение PROTOCOL между Benign и Attack


PROTOCOL,Benign protocol percentage,Attack protocol percentage
i8,f64,f64
6,49.046825,95.629256
58,0.005536,null
1,0.030043,0.015779
17,50.911729,4.350379
2,0.005847,0.004586
47,0.00002,null


PROTOCOL,Benign protocol percentage
i8,f64
6,49.046825
1,0.030043
58,0.005536
2,0.005847
17,50.911729
47,0.00002


In [87]:
df.group_by(pl.col('is_attack')).agg(pl.col('IN_BYTES').mean(), pl.col('OUT_BYTES').mean(), pl.col('FLOW_DURATION_MILLISECONDS').mean(), pl.col('Label').count().alias('Count'))

is_attack,IN_BYTES,OUT_BYTES,FLOW_DURATION_MILLISECONDS,Count
i32,f64,f64,f64,u32
0,867.640543,8253.537161,185715.810107,15101685
1,10013.788006,2301.412192,3.7472e6,2028030


In [98]:
df = df.with_columns(
    pl.when((pl.col('IN_BYTES') / (pl.col('OUT_BYTES')+1) > 10) & (pl.col('FLOW_DURATION_MILLISECONDS') < 500) & (pl.col('IN_PKTS') > 10))
                     .then(1)
                     .otherwise(0)
                     .alias('is_suspicious')
)

df_filtered = df.filter(pl.col('is_suspicious') == 1)
total_flagged = df_filtered.shape[0]
true_attacks = df_filtered.filter(pl.col('Label') != 0).shape[0]
print('Записей помечено как is_suspicious == 1:', total_flagged)
print('Из них на самом деле являются атаками:', true_attacks)
print('Точность эвристики: ', true_attacks / total_flagged)

Записей помечено как is_suspicious == 1: 4645
Из них на самом деле являются атаками: 3707
Точность эвристики:  0.7980624327233584
